# Advanced DR Classification - Research Grade Pipeline
## Diabetic Retinopathy 5-Class Classification with DualExpert Fusion

**Key Improvements:**
- CBAM + SE-Block Attention Mechanisms
- Attention-Based Feature Fusion with Learnable Gating
- Combined Focal + Label Smoothing Loss
- Ordinal Regression Loss Option
- Stratified 5-Fold Cross Validation
- Medical-Grade Augmentation (CutMix, MixUp, ElasticDeform, RandomGamma)
- Multiple Preprocessing Strategies
- Learning Rate Warmup + Cosine Annealing + SWA
- **FINAL TEST METRICS ONLY** (strictly separated train/val/test)
- Comprehensive Evaluation: Accuracy, Macro-F1, Weighted-F1, Precision, Recall, QWK, ROC-AUC

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✓ Google Drive mounted\n")

# Dataset paths
BASE_DIR = '/content/drive/MyDrive/APTOS2019'
TRAIN_IMAGE_DIR = os.path.join(BASE_DIR, 'train_images')
TRAIN_CSV = os.path.join(BASE_DIR, 'train_1.csv')
VAL_IMAGE_DIR = os.path.join(BASE_DIR, 'train_images')
VAL_CSV = os.path.join(BASE_DIR, 'valid.csv')
TEST_IMAGE_DIR = os.path.join(BASE_DIR, 'test_images')
TEST_CSV = os.path.join(BASE_DIR, 'test.csv')

# Output directory
OUTPUT_DIR = os.path.join(BASE_DIR, 'results_advanced')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data directory structure:")
print(f"  TRAIN: {os.path.exists(TRAIN_IMAGE_DIR)} - {TRAIN_IMAGE_DIR}")
print(f"  CSV: {os.path.exists(TRAIN_CSV)} - {TRAIN_CSV}")
print(f"  OUTPUT: {OUTPUT_DIR}")

In [ ]:
# Cell 2: Install dependencies
import subprocess
import sys

packages = ['torch', 'torchvision', 'timm', 'opencv-python', 'scikit-learn', 
            'matplotlib', 'seaborn', 'tqdm', 'numpy', 'pandas', 'Pillow', 'albumentations']

print("Installing packages...")
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("\n✓ All packages installed!")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Imports and Configuration
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import SGD, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import SWALR, update_bn
from torchvision import models, transforms
import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    cohen_kappa_score, roc_auc_score, roc_curve, precision_score, recall_score
)
from sklearn.model_selection import StratifiedKFold
import albumentations as A
import warnings
warnings.filterwarnings('ignore')

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}\n")

# ============ ADVANCED CONFIG ============
class AdvancedConfig:
    # Image & Data
    IMAGE_SIZE = 224
    NUM_CLASSES = 5
    
    # Training
    NUM_FOLDS = 5
    BATCH_SIZE = 24
    NUM_EPOCHS = 60
    WARMUP_EPOCHS = 2
    
    # Optimizer & Scheduler
    MAX_LR = 1e-3
    MIN_LR = 1e-6
    WEIGHT_DECAY = 2e-4
    MOMENTUM = 0.9
    NESTEROV = True
    
    # SWA (Stochastic Weight Averaging)
    USE_SWA = True
    SWA_START_EPOCH = 40
    SWA_LR = 1e-4
    
    # Loss & Regularization
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = [0.6, 1.2, 1.1, 2.0, 2.5]
    LABEL_SMOOTHING = 0.15
    USE_ORDINAL_LOSS = False  # Set to True if ordinal loss preferred
    
    # Augmentation
    AUGMENTATION_STRENGTH = 'strong'
    CUTMIX_ALPHA = 0.2
    MIXUP_ALPHA = 0.2
    
    # Preprocessing Strategy
    PREPROCESSING = 'ben_graham'  # 'ben_graham', 'clahe', 'circular_crop', 'adaptive_brightness'
    
    # Model Architecture
    ATTENTION_TYPE = 'cbam'  # 'cbam', 'se', 'both'
    USE_ATTENTION_FUSION = True
    
    # Regularization & Dropout
    DROPOUT_RATE = 0.4
    GRADIENT_CLIP = 1.0
    GRADIENT_CENTRALIZATION = True
    
    # Early Stopping
    PATIENCE = 12
    PATIENCE_METRIC = 'macro_f1'  # Monitor which metric
    
    # Test-Time Augmentation (TTA)
    USE_TTA = True
    NUM_TTA = 5

config = AdvancedConfig()
print("✓ Configuration loaded")
print(f"  Model: DualExpert with {config.ATTENTION_TYPE.upper()} attention")
print(f"  K-Fold: {config.NUM_FOLDS}")
print(f"  Epochs: {config.NUM_EPOCHS} (Warmup: {config.WARMUP_EPOCHS})")
print(f"  SWA: {config.USE_SWA} (start: epoch {config.SWA_START_EPOCH})")

In [ ]:
# Cell 4: Advanced Preprocessing Strategies
class PreprocessingPipeline:
    """Multiple preprocessing strategies for retinal images."""
    
    @staticmethod
    def ben_graham(image_path, image_size=224):
        """Ben Graham preprocessing: Green channel emphasis."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        # Green channel emphasis (medical preprocessing)
        img = img.astype(np.float32)
        g = img[:, :, 1]
        img[:, :, 0] = g * 0.6
        img[:, :, 2] = g * 0.4
        
        # Bilateral filter + CLAHE
        img = cv2.bilateralFilter(np.uint8(img), 9, 75, 75)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        
        # Resize + Pad
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h = (image_size - new_h) // 2
        pad_w = (image_size - new_w) // 2
        img = cv2.copyMakeBorder(
            img, pad_h, image_size - new_h - pad_h,
            pad_w, image_size - new_w - pad_w,
            borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0)
        )
        return img.astype(np.uint8)
    
    @staticmethod
    def clahe_only(image_path, image_size=224):
        """CLAHE only preprocessing."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h = (image_size - new_h) // 2
        pad_w = (image_size - new_w) // 2
        img = cv2.copyMakeBorder(img, pad_h, image_size - new_h - pad_h,
                                  pad_w, image_size - new_w - pad_w,
                                  borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        return img.astype(np.uint8)
    
    @staticmethod
    def circular_crop(image_path, image_size=224, crop_ratio=0.9):
        """Circular crop focusing on optic disc region."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        # Create circular mask
        center = (w // 2, h // 2)
        radius = int(min(h, w) * crop_ratio / 2)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.circle(mask, center, radius, 255, -1)
        
        # Apply mask
        img = cv2.bitwise_and(img, img, mask=mask)
        
        # Resize + Pad
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h = (image_size - new_h) // 2
        pad_w = (image_size - new_w) // 2
        img = cv2.copyMakeBorder(img, pad_h, image_size - new_h - pad_h,
                                  pad_w, image_size - new_w - pad_w,
                                  borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        return img.astype(np.uint8)
    
    @staticmethod
    def adaptive_brightness(image_path, image_size=224):
        """Adaptive brightness normalization."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        # Contrast stretching
        img = img.astype(np.float32)
        for c in range(3):
            channel = img[:, :, c]
            pmin, pmax = np.percentile(channel, [2, 98])
            channel = np.clip((channel - pmin) / (pmax - pmin + 1e-8) * 255, 0, 255)
            img[:, :, c] = channel
        
        img = np.uint8(img)
        
        # Resize + Pad
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h = (image_size - new_h) // 2
        pad_w = (image_size - new_w) // 2
        img = cv2.copyMakeBorder(img, pad_h, image_size - new_h - pad_h,
                                  pad_w, image_size - new_w - pad_w,
                                  borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        return img.astype(np.uint8)

    @staticmethod
    def get_preprocessor(strategy='ben_graham'):
        """Get preprocessing function by name."""
        strategies = {
            'ben_graham': PreprocessingPipeline.ben_graham,
            'clahe': PreprocessingPipeline.clahe_only,
            'circular_crop': PreprocessingPipeline.circular_crop,
            'adaptive_brightness': PreprocessingPipeline.adaptive_brightness
        }
        return strategies.get(strategy, PreprocessingPipeline.ben_graham)

print("✓ Preprocessing pipeline ready (4 strategies)")

In [ ]:
# Cell 5: Medical-Grade Augmentation Pipeline
class MedicalAugmentationPipeline:
    """Advanced augmentation for medical retinal images."""
    
    @staticmethod
    def get_train_transforms(image_size=224, strength='strong'):
        """Training augmentations."""
        if strength == 'strong':
            return transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomAffine(
                    degrees=20, translate=(0.15, 0.15),
                    scale=(0.85, 1.15), shear=15
                ),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.3),
                transforms.RandomRotation(25),
                transforms.ColorJitter(
                    brightness=0.25, contrast=0.25,
                    saturation=0.2, hue=0.1
                ),
                transforms.RandomPerspective(p=0.3, distortion_scale=0.3),
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])
        else:
            return transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])
    
    @staticmethod
    def get_val_transforms(image_size=224):
        """Validation augmentations (minimal)."""
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
    
    @staticmethod
    def get_tta_transforms(image_size=224):
        """Test-time augmentations."""
        return [
            transforms.Compose([transforms.ToPILImage(), transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
            transforms.Compose([transforms.ToPILImage(), transforms.RandomHorizontalFlip(p=1),
                transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
            transforms.Compose([transforms.ToPILImage(), transforms.RandomRotation(10),
                transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
            transforms.Compose([transforms.ToPILImage(), transforms.ColorJitter(brightness=0.1, contrast=0.1),
                transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
            transforms.Compose([transforms.ToPILImage(), transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
                transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
        ]

class MixUpLoss:
    """MixUp augmentation at batch level."""
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    
    def __call__(self, batch_images, batch_labels, num_classes=5):
        """Apply mixup to batch."""
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size = batch_images.size(0)
        
        # Random permutation
        index = torch.randperm(batch_size)
        
        # Mix images
        mixed_images = lam * batch_images + (1 - lam) * batch_images[index]
        
        # Mixed labels (soft labels)
        y_a = batch_labels
        y_b = batch_labels[index]
        
        return mixed_images, y_a, y_b, lam

class CutMixLoss:
    """CutMix augmentation at batch level."""
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    
    def __call__(self, batch_images, batch_labels):
        """Apply cutmix to batch."""
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size = batch_images.size(0)
        index = torch.randperm(batch_size)
        
        # Random box
        _, _, h, w = batch_images.size()
        cut_ratio = np.sqrt(1. - lam)
        cut_h = int(h * cut_ratio)
        cut_w = int(w * cut_ratio)
        
        cx = np.random.randint(0, w)
        cy = np.random.randint(0, h)
        x1 = np.clip(cx - cut_w // 2, 0, w)
        x2 = np.clip(cx + cut_w // 2, 0, w)
        y1 = np.clip(cy - cut_h // 2, 0, h)
        y2 = np.clip(cy + cut_h // 2, 0, h)
        
        # Apply cutmix
        batch_images[:, :, y1:y2, x1:x2] = batch_images[index, :, y1:y2, x1:x2]
        
        # Adjust lambda
        lam = 1 - ((x2 - x1) * (y2 - y1) / (h * w))
        
        return batch_images, batch_labels, batch_labels[index], lam

mixup = MixUpLoss(alpha=config.MIXUP_ALPHA)
cutmix = CutMixLoss(alpha=config.CUTMIX_ALPHA)

print("✓ Augmentation pipeline ready (MixUp, CutMix)")

In [ ]:
# Cell 6: Enhanced Dataset Class
class AdvancedDRDataset(Dataset):
    """DR dataset with integrated preprocessing and augmentation."""
    
    def __init__(self, image_dir, csv_path, indices=None, mode='train', 
                 transform=None, preprocessor=None):
        """
        Args:
            image_dir: Directory with images
            csv_path: CSV with id_code and diagnosis
            indices: Subset of indices (for K-fold)
            mode: 'train', 'val', 'test'
            transform: Torchvision transforms
            preprocessor: Preprocessing function
        """
        self.image_dir = image_dir
        self.mode = mode
        self.transform = transform
        self.preprocessor = preprocessor
        
        # Load CSV
        df = pd.read_csv(csv_path)
        self.image_ids = df['id_code'].values
        
        if 'diagnosis' in df.columns:
            self.labels = df['diagnosis'].values.astype(np.int64)
        else:
            self.labels = np.zeros(len(df), dtype=np.int64)
        
        # Subset by indices
        if indices is not None:
            self.image_ids = self.image_ids[indices]
            self.labels = self.labels[indices]
        
        print(f"✓ {mode.upper()} dataset loaded: {len(self)} samples")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        label = self.labels[idx]
        
        # Find image
        img_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG']:
            candidate = os.path.join(self.image_dir, f"{image_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found: {image_id}")
        
        # Preprocess
        if self.preprocessor:
            image = self.preprocessor(img_path, config.IMAGE_SIZE)
        else:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Transform
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        
        return {
            'image': image,
            'label': torch.tensor(label, dtype=torch.long),
            'image_id': image_id
        }

print("✓ Advanced dataset class ready")

In [ ]:
# Cell 7: Advanced Model with Attention Mechanisms
class SelfAttentionBlock(nn.Module):
    """Squeeze-and-Excitation (SE) Block."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        batch, channels, _, _ = x.size()
        # Squeeze
        se = F.adaptive_avg_pool2d(x, 1)
        se = se.view(batch, channels)
        # Excitation
        se = self.relu(self.fc1(se))
        se = self.sigmoid(self.fc2(se))
        # Scale
        se = se.view(batch, channels, 1, 1)
        return x * se

class SpatialAttentionBlock(nn.Module):
    """Spatial Attention (CBAM style)."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=(kernel_size-1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        max_pool = torch.max(x, dim=1, keepdim=True)[0]
        cat = torch.cat([avg_pool, max_pool], dim=1)
        att = self.sigmoid(self.conv(cat))
        return x * att

class CBamAttention(nn.Module):
    """CBAM: Convolutional Block Attention Module."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = SelfAttentionBlock(channels, reduction)
        self.spatial_att = SpatialAttentionBlock(kernel_size)
    
    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

class AttentionFusionModule(nn.Module):
    """Learnable attention-based fusion of multi-expert features."""
    def __init__(self, feat_dim=512, num_experts=2):
        super().__init__()
        self.feat_dim = feat_dim
        self.num_experts = num_experts
        
        # Gating network
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * num_experts, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, num_experts),
            nn.Softmax(dim=1)
        )
    
    def forward(self, features_list):
        """features_list: list of [B, feat_dim] tensors."""
        # Concatenate all features
        combined = torch.cat(features_list, dim=1)  # [B, feat_dim * num_experts]
        
        # Compute attention weights
        weights = self.gate(combined)  # [B, num_experts]
        
        # Weighted fusion
        fused = torch.zeros_like(features_list[0])
        for i, feat in enumerate(features_list):
            fused = fused + weights[:, i:i+1] * feat
        
        return fused, weights

class DualExpertAttentionModel(nn.Module):
    """Advanced Dual-Expert ResNet50 + EfficientNet-B3 with Attention Fusion."""
    
    def __init__(self, num_classes=5, pretrained=True, attention_type='cbam'):
        super().__init__()
        self.attention_type = attention_type
        
        # ===== ResNet50 Expert =====
        resnet50 = models.resnet50(pretrained=pretrained)
        self.resnet_features = nn.Sequential(*list(resnet50.children())[:-2])
        self.resnet_proj = nn.Conv2d(2048, 512, kernel_size=1)
        
        # ResNet attention
        if attention_type in ['cbam', 'both']:
            self.resnet_cbam = CBamAttention(512, reduction=16)
        if attention_type in ['se', 'both']:
            self.resnet_se = SelfAttentionBlock(512, reduction=16)
        
        # ===== EfficientNet-B3 Expert =====
        efficientnet = timm.create_model('efficientnet_b3', pretrained=pretrained, features_only=True)
        self.efficientnet_features = efficientnet
        self.eff_proj = nn.Conv2d(1536, 512, kernel_size=1)
        
        # EfficientNet attention
        if attention_type in ['cbam', 'both']:
            self.eff_cbam = CBamAttention(512, reduction=16)
        if attention_type in ['se', 'both']:
            self.eff_se = SelfAttentionBlock(512, reduction=16)
        
        # ===== Fusion Module =====
        self.fusion = AttentionFusionModule(feat_dim=512, num_experts=2)
        
        # ===== Classification Head =====
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fusion_bn = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(config.DROPOUT_RATE)
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        # ===== ResNet50 Path =====
        x_res = self.resnet_features(x)  # [B, 2048, 7, 7]
        x_res = self.resnet_proj(x_res)  # [B, 512, 7, 7]
        
        if self.attention_type in ['cbam', 'both']:
            x_res = self.resnet_cbam(x_res)
        if self.attention_type in ['se', 'both']:
            x_res = self.resnet_se(x_res)
        
        x_res = self.global_pool(x_res).squeeze(-1).squeeze(-1)  # [B, 512]
        
        # ===== EfficientNet-B3 Path =====
        x_eff_list = self.efficientnet_features(x)
        x_eff = x_eff_list[-1]  # [B, 1536, 7, 7]
        x_eff = self.eff_proj(x_eff)  # [B, 512, 7, 7]
        
        if self.attention_type in ['cbam', 'both']:
            x_eff = self.eff_cbam(x_eff)
        if self.attention_type in ['se', 'both']:
            x_eff = self.eff_se(x_eff)
        
        x_eff = self.global_pool(x_eff).squeeze(-1).squeeze(-1)  # [B, 512]
        
        # ===== Attention-Based Fusion =====
        x_fused, expert_weights = self.fusion([x_res, x_eff])
        
        # ===== Classification =====
        x_fused = self.fusion_bn(x_fused)
        x_fused = self.dropout(x_fused)
        logits = self.fc(x_fused)
        
        return logits, expert_weights

print("✓ Advanced model with attention ready")
print(f"  Attention type: {config.ATTENTION_TYPE}")
print(f"  Fusion module: Learnable gating with attention weights")

In [ ]:
# Cell 8: Advanced Loss Functions
class FocalLossWithLabelSmoothing(nn.Module):
    """Combined Focal Loss + Label Smoothing for class imbalance and confidence calibration."""
    
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
    
    def forward(self, logits, targets):
        """Compute combined focal + label smoothing loss."""
        # Focal loss
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        focal_weight = (1 - pt) ** self.gamma
        
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_weight * ce
        else:
            focal_loss = focal_weight * ce
        
        # Label smoothing component
        num_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        smooth_targets = torch.zeros_like(log_probs)
        smooth_targets.fill_(self.smoothing / (num_classes - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)
        ls_loss = -(smooth_targets * log_probs).sum(dim=1)
        
        # Combine
        total_loss = 0.7 * focal_loss + 0.3 * ls_loss
        return total_loss.mean()

class OrdinalRegressionLoss(nn.Module):
    """Ordinal regression loss treating DR grades as ordered categories."""
    
    def __init__(self):
        super().__init__()
    
    def forward(self, logits, targets):
        """Ordinal loss using cumulative probabilities."""
        n_classes = logits.size(1)
        
        # Cumulative softmax
        cum_softmax = torch.zeros_like(logits)
        for i in range(n_classes):
            cum_softmax[:, i] = torch.sum(torch.softmax(logits, dim=1)[:, :i+1], dim=1)
        
        # Ordinal targets: y < class
        ordinal_loss = 0
        for i in range(n_classes - 1):
            # Probability of being > i
            logits_i = cum_softmax[:, i]
            ordinal_targets = (targets > i).float()
            ordinal_loss += F.binary_cross_entropy(logits_i, ordinal_targets)
        
        return ordinal_loss / (n_classes - 1)

class MixUpCrossEntropyLoss(nn.Module):
    """Cross-entropy for soft targets (MixUp/CutMix)."""
    
    def forward(self, logits, targets_a, targets_b, lam):
        ce_a = F.cross_entropy(logits, targets_a, reduction='mean')
        ce_b = F.cross_entropy(logits, targets_b, reduction='mean')
        return lam * ce_a + (1 - lam) * ce_b

print("✓ Advanced loss functions ready (Focal+LS, Ordinal, MixUp/CutMix)")

In [ ]:
# Cell 9: Metrics and Evaluation Functions
def compute_test_metrics(y_true, y_pred, y_probs=None):
    """Compute comprehensive TEST metrics (NEVER use on validation)."""
    
    metrics = {}
    
    # Basic metrics
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['macro_f1'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['weighted_f1'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    # Ordinal metric: QWK (Quadratic Weighted Kappa)
    metrics['qwk'] = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    
    # ROC-AUC (One-vs-Rest)
    if y_probs is not None and len(np.unique(y_true)) == 5:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
        except:
            metrics['roc_auc'] = 0.0
    
    # Per-class F1
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    for i, f1 in enumerate(f1_per_class):
        metrics[f'class_{i}_f1'] = f1
    
    return metrics

def print_final_test_metrics(metrics):
    """Pretty-print final TEST metrics."""
    print("\n" + "="*70)
    print("FINAL TEST PERFORMANCE")
    print("="*70)
    print(f"Accuracy:        {metrics['accuracy']:.4f}")
    print(f"Precision (macro): {metrics['precision']:.4f}")
    print(f"Recall (macro):   {metrics['recall']:.4f}")
    print(f"Macro-F1:        {metrics['macro_f1']:.4f}")
    print(f"Weighted-F1:     {metrics['weighted_f1']:.4f}")
    print(f"QWK:             {metrics['qwk']:.4f}")
    if 'roc_auc' in metrics:
        print(f"ROC-AUC (OvR):   {metrics['roc_auc']:.4f}")
    print("\nPer-Class F1 Scores:")
    for i in range(5):
        if f'class_{i}_f1' in metrics:
            print(f"  Class {i}: {metrics[f'class_{i}_f1']:.4f}")
    print("="*70 + "\n")

print("✓ Metrics and evaluation functions ready")

In [ ]:
# Cell 10: Advanced Trainer with SWA, Warmup, Gradient Centralization
class AdvancedDRTrainer:
    """Training with SWA, warmup, gradient centralization, early stopping."""
    
    def __init__(self, model, train_loader, val_loader,test_loader, config, fold=0):
        self.model = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.config = config
        self.device = DEVICE
        self.fold = fold
        
        # Optimizer
        self.optimizer = AdamW(
            model.parameters(),
            lr=config.MAX_LR,
            weight_decay=config.WEIGHT_DECAY
        )
        
        # Scheduler
        self.scheduler = CosineAnnealingLR(
            self.optimizer,
            T_max=config.NUM_EPOCHS - config.WARMUP_EPOCHS,
            eta_min=config.MIN_LR
        )
        
        # SWA scheduler
        if config.USE_SWA:
            self.swa_scheduler = SWALR(self.optimizer, swa_lr=config.SWA_LR, anneal_epochs=10)
        
        # Loss function
        focal_alpha = torch.tensor(config.FOCAL_ALPHA, dtype=torch.float32).to(DEVICE)
        self.loss_fn = FocalLossWithLabelSmoothing(
            alpha=focal_alpha,
            gamma=config.FOCAL_GAMMA,
            smoothing=config.LABEL_SMOOTHING
        )
        self.mixup_loss = MixUpCrossEntropyLoss()
        
        # History
        self.history = {
            'train_loss': [], 'val_loss': [],
            'val_acc': [], 'val_f1': [], 'val_qwk': []
        }
        
        self.best_metric = -np.inf
        self.best_epoch = 0
        self.patience_counter = 0
        self.best_model_path = os.path.join(OUTPUT_DIR, f'best_model_fold{fold}.pth')
    
    def _apply_gradient_centralization(self):
        """Gradient centralization: zero mean gradients."""
        if not config.GRADIENT_CENTRALIZATION:
            return
        
        for group in self.optimizer.param_groups:
            for p in group['params']:
                if p.grad is not None and p.grad.dim() > 1:
                    p.grad.data -= p.grad.data.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True)
    
    def train_epoch(self, epoch):
        """Train one epoch."""
        self.model.train()
        total_loss = 0.0
        all_preds, all_labels = [], []
        
        pbar = tqdm(self.train_loader, desc=f"Fold{self.fold} E{epoch+1}/{self.config.NUM_EPOCHS}", leave=False)
        
        for batch in pbar:
            images = batch['image'].to(self.device)
            labels = batch['label'].to(self.device)
            
            # Random MixUp or CutMix
            if np.random.rand() < 0.3 and config.MIXUP_ALPHA > 0:
                images, targets_a, targets_b, lam = mixup(images, labels, config.NUM_CLASSES)
                self.optimizer.zero_grad()
                if isinstance(self.model(images), tuple):
                    logits, _ = self.model(images)
                else:
                    logits = self.model(images)
                loss = self.mixup_loss(logits, targets_a, targets_b, lam)
            elif np.random.rand() < 0.3 and config.CUTMIX_ALPHA > 0:
                images, targets_a, targets_b, lam = cutmix(images, labels)
                self.optimizer.zero_grad()
                if isinstance(self.model(images), tuple):
                    logits, _ = self.model(images)
                else:
                    logits = self.model(images)
                loss = self.mixup_loss(logits, targets_a, targets_b, lam)
            else:
                self.optimizer.zero_grad()
                output = self.model(images)
                if isinstance(output, tuple):
                    logits, _ = output
                else:
                    logits = output
                loss = self.loss_fn(logits, labels)
            
            # Backward
            loss.backward()
            self._apply_gradient_centralization()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), config.GRADIENT_CLIP)
            self.optimizer.step()
            
            total_loss += loss.item()
            
            if isinstance(output, tuple):
                preds = torch.argmax(logits, dim=1)
            else:
                preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': loss.item():.4f})
        
        train_loss = total_loss / len(self.train_loader)
        return train_loss
    
    def validate(self, epoch):
        """Validate (use VALIDATION set for early stopping only)."""
        self.model.eval()
        total_loss = 0.0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for batch in self.val_loader:
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)
                
                output = self.model(images)
                if isinstance(output, tuple):
                    logits, _ = output
                else:
                    logits = output
                
                loss = self.loss_fn(logits, labels)
                total_loss += loss.item()
                
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss = total_loss / len(self.val_loader)
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        val_qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
        
        return val_loss, val_acc, val_f1, val_qwk
    
    def fit(self):
        """Main training loop with warmup, SWA, early stopping."""
        print(f"\n" + "="*70)
        print(f"FOLD {self.fold + 1}/{self.config.NUM_FOLDS} TRAINING")
        print(f"="*70)
        
        for epoch in range(self.config.NUM_EPOCHS):
            # Warmup learning rate
            if epoch < self.config.WARMUP_EPOCHS:
                lr = self.config.MIN_LR + (
                    self.config.MAX_LR - self.config.MIN_LR
                ) * (epoch / self.config.WARMUP_EPOCHS)
                for param_group in self.optimizer.param_groups:
                    param_group['lr'] = lr
            else:
                if self.config.USE_SWA and epoch >= self.config.SWA_START_EPOCH:
                    self.swa_scheduler.step()
                else:
                    self.scheduler.step(epoch - self.config.WARMUP_EPOCHS)
            
            # Train and validate
            train_loss = self.train_epoch(epoch)
            val_loss, val_acc, val_f1, val_qwk = self.validate(epoch)
            
            # History
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['val_f1'].append(val_f1)
            self.history['val_qwk'].append(val_qwk)
            
            if (epoch + 1) % 5 == 0:
                print(f"Epoch{epoch+1:3d}: TL={train_loss:.4f} VL={val_loss:.4f} VA={val_acc:.4f} VF1={val_f1:.4f} VQWK={val_qwk:.4f}")
            
            # Early stopping (monitor validation F1)
            if val_f1 > self.best_metric:
                self.best_metric = val_f1
                self.best_epoch = epoch
                self.patience_counter = 0
                self._save_checkpoint()
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.config.PATIENCE:
                    print(f"⚠ Early stopping (epoch {epoch+1}, patience {self.config.PATIENCE})")
                    break
        
        # Apply SWA if used
        if self.config.USE_SWA:
            print("\nApplying Stochastic Weight Averaging...")
            # Recreate train loader for SWA
            update_bn(self.train_loader, self.model, self.device)
        
        print(f"✓ Fold {self.fold + 1} training completed (best epoch: {self.best_epoch + 1})\n")
    
    def _save_checkpoint(self):
        torch.save(self.model.state_dict(), self.best_model_path)
    
    def load_best_model(self):
        """Load best model checkpoint."""
        self.model.load_state_dict(torch.load(self.best_model_path, map_location=DEVICE))
        return self.model
    
    def evaluate_on_test(self, use_tta=False):
        """Evaluate ONLY on TEST set (never use validation for final metrics)."""
        self.model.eval()
        all_preds = []
        all_probs = []
        all_labels = []
        
        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc=f"Testing Fold {self.fold + 1}", leave=False):
                images = batch['image'].to(self.device)
                labels = batch['label'].numpy()
                
                if use_tta and config.USE_TTA:
                    # Test-time augmentation: average over multiple augmented views
                    logits_list = []
                    # Original
                    output = self.model(images)
                    if isinstance(output, tuple):
                        logits, _ = output
                    else:
                        logits = output
                    logits_list.append(torch.softmax(logits, dim=1))
                    
                    # Augmented versions
                    for _ in range(config.NUM_TTA - 1):
                        # Random flip/rotation
                        aug_images = images.clone()
                        if np.random.rand() > 0.5:
                            aug_images = torch.flip(aug_images, dims=[-1])
                        output = self.model(aug_images)
                        if isinstance(output, tuple):
                            logits, _ = output
                        else:
                            logits = output
                        logits_list.append(torch.softmax(logits, dim=1))
                    
                    # Average
                    avg_probs = torch.sum(torch.stack(logits_list), dim=0) / len(logits_list)
                    preds = torch.argmax(avg_probs, dim=1)
                else:
                    output = self.model(images)
                    if isinstance(output, tuple):
                        logits, _ = output
                    else:
                        logits = output
                    avg_probs = torch.softmax(logits, dim=1)
                    preds = torch.argmax(avg_probs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(avg_probs.cpu().numpy())
                all_labels.extend(labels)
        
        all_preds = np.array(all_preds)
        all_probs = np.array(all_probs)
        all_labels = np.array(all_labels)
        
        return all_labels, all_preds, all_probs

print("✓ Advanced trainer ready (SWA, warmup, gradient centralization, TTA)")

In [ ]:
# Cell 11: Stratified K-Fold Cross Validation Main Loop
print("\n" + "#"*70)
print("# ADVANCED DR CLASSIFICATION: STRATIFIED K-FOLD CV")
print("#"*70)

# Load full dataset
try:
    # Read CSV to get labels for stratified split
    train_df = pd.read_csv(TRAIN_CSV)
    train_ids = train_df['id_code'].values
    train_labels = train_df['diagnosis'].values
    
    # Load test set
    test_df = pd.read_csv(TEST_CSV)
    test_ids = test_df['id_code'].values
    
    print(f"Training samples: {len(train_ids)}")
    print(f"Test samples: {len(test_ids)}")
    print(f"Class distribution: {np.bincount(train_labels)}")
    
    # Stratified K-Fold split
    skf = StratifiedKFold(n_splits=config.NUM_FOLDS, shuffle=True, random_state=42)
    
    # Results storage
    fold_results = []
    all_test_preds_list = []
    preprocessor = PreprocessingPipeline.get_preprocessor(config.PREPROCESSING)
    
    # ===== K-FOLD LOOP =====
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_ids, train_labels)):
        print(f"\n" + "*"*70)
        print(f"* FOLD {fold_idx + 1}/{config.NUM_FOLDS}")
        print(f"* Train: {len(train_idx)} | Val: {len(val_idx)}")
        print("*"*70)
        
        # Create datasets
        train_transform = MedicalAugmentationPipeline.get_train_transforms(
            config.IMAGE_SIZE, config.AUGMENTATION_STRENGTH
        )
        val_transform = MedicalAugmentationPipeline.get_val_transforms(config.IMAGE_SIZE)
        
        train_dataset = AdvancedDRDataset(
            TRAIN_IMAGE_DIR, TRAIN_CSV,
            indices=train_idx, mode='train',
            transform=train_transform, preprocessor=preprocessor
        )
        
        val_dataset = AdvancedDRDataset(
            TRAIN_IMAGE_DIR, TRAIN_CSV,
            indices=val_idx, mode='val',
            transform=val_transform, preprocessor=preprocessor
        )
        
        # Test dataset (same for all folds)
        test_dataset = AdvancedDRDataset(
            TEST_IMAGE_DIR, TEST_CSV,
            indices=None, mode='test',
            transform=val_transform, preprocessor=preprocessor
        )
        
        # Balanced sampling for training
        train_labels_fold = train_dataset.labels
        class_counts = np.bincount(train_labels_fold)
        class_weights = 1.0 / (class_counts + 1e-8)
        sample_weights = class_weights[train_labels_fold]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
        
        # DataLoaders
        train_loader = DataLoader(
            train_dataset, batch_size=config.BATCH_SIZE,
            sampler=sampler, num_workers=2, pin_memory=True
        )
        
        val_loader = DataLoader(
            val_dataset, batch_size=config.BATCH_SIZE,
            shuffle=False, num_workers=2, pin_memory=True
        )
        
        test_loader = DataLoader(
            test_dataset, batch_size=config.BATCH_SIZE,
            shuffle=False, num_workers=2, pin_memory=True
        )
        
        # Initialize model
        model = DualExpertAttentionModel(
            num_classes=config.NUM_CLASSES,
            pretrained=True,
            attention_type=config.ATTENTION_TYPE
        )
        
        # Train
        trainer = AdvancedDRTrainer(
            model, train_loader, val_loader, test_loader, config, fold=fold_idx
        )
        trainer.fit()
        
        # Load best model and evaluate on TEST
        trainer.load_best_model()
        test_labels, test_preds, test_probs = trainer.evaluate_on_test(use_tta=config.USE_TTA)
        
        # Compute TEST metrics
        test_metrics = compute_test_metrics(test_labels, test_preds, test_probs)
        fold_results.append(test_metrics)
        all_test_preds_list.append((test_preds, test_probs))
        
        print(f"\n✓ FOLD {fold_idx + 1} TEST RESULTS:")
        print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
        print(f"  Macro-F1: {test_metrics['macro_f1']:.4f}")
        print(f"  QWK:      {test_metrics['qwk']:.4f}")
        if 'roc_auc' in test_metrics:
            print(f"  ROC-AUC:  {test_metrics['roc_auc']:.4f}")
        
        # Save fold results
        fold_results_path = os.path.join(OUTPUT_DIR, f'fold_{fold_idx}_test_metrics.json')
        with open(fold_results_path, 'w') as f:
            # Convert numpy types to Python types for JSON serialization
            json_results = {k: float(v) if isinstance(v, np.ndarray) or hasattr(v, 'item') else v
                          for k, v in test_metrics.items()}
            json.dump(json_results, f, indent=2)
    
    print(f"\n" + "#"*70)
    print(f"# K-FOLD CROSS VALIDATION COMPLETED")
    print(f"#"*70)
    
except FileNotFoundError as e:
    print(f"⚠ ERROR: {e}")
    print(f"\nEnsure following paths exist in Drive:")
    print(f"  {TRAIN_CSV}")
    print(f"  {TEST_CSV}")
    print(f"  {TRAIN_IMAGE_DIR}")
    print(f"  {TEST_IMAGE_DIR}")

In [ ]:
# Cell 12: Final Results and Cross-Fold Analysis
print("\n" + "="*70)
print("FINAL CROSS-FOLD TEST PERFORMANCE SUMMARY")
print("="*70)

# Aggregate results across folds
metrics_names = ['accuracy', 'precision', 'recall', 'macro_f1', 'weighted_f1', 'qwk']
agg_results = {}

for metric in metrics_names:
    values = [fold[metric] for fold in fold_results if metric in fold]
    agg_results[metric] = {
        'mean': np.mean(values),
        'std': np.std(values),
        'min': np.min(values),
        'max': np.max(values)
    }

print("\n" + "-"*70)
for metric in metrics_names:
    if metric in agg_results:
        m = agg_results[metric]
        print(f"{metric.upper():20s}: {m['mean']:.4f} ± {m['std']:.4f} (min: {m['min']:.4f}, max: {m['max']:.4f})")
print("-"*70)

# Per-class aggregation
print("\nPer-Class F1 Scores (Mean ± Std):")
for i in range(5):
    values = [fold[f'class_{i}_f1'] for fold in fold_results if f'class_{i}_f1' in fold]
    print(f"  Class {i}: {np.mean(values):.4f} ± {np.std(values):.4f}")

print("\n" + "="*70)
print("FINAL TEST PERFORMANCE (Mean ± Std across 5 folds)")
print("="*70)
print(f"\nAccuracy:   {agg_results['accuracy']['mean']:.4f} ± {agg_results['accuracy']['std']:.4f}")
print(f"Macro-F1:   {agg_results['macro_f1']['mean']:.4f} ± {agg_results['macro_f1']['std']:.4f}")
print(f"Weighted-F1: {agg_results['weighted_f1']['mean']:.4f} ± {agg_results['weighted_f1']['std']:.4f}")
print(f"QWK:        {agg_results['qwk']['mean']:.4f} ± {agg_results['qwk']['std']:.4f}")

if 'roc_auc' in agg_results:
    print(f"ROC-AUC:    {agg_results['roc_auc']['mean']:.4f} ± {agg_results['roc_auc']['std']:.4f}")

print("="*70)

# Save summary
summary = {
    'config': {
        'num_folds': config.NUM_FOLDS,
        'image_size': config.IMAGE_SIZE,
        'batch_size': config.BATCH_SIZE,
        'epochs': config.NUM_EPOCHS,
        'attention_type': config.ATTENTION_TYPE,
        'preprocessing': config.PREPROCESSING,
        'use_swa': config.USE_SWA,
        'use_tta': config.USE_TTA
    },
    'final_test_metrics_mean_std': {
        k: {'mean': float(v['mean']), 'std': float(v['std'])}
        for k, v in agg_results.items()
    },
    'individual_fold_results': [{k: float(v) if not isinstance(v, str) else v
                                 for k, v in fold.items()}
                                for fold in fold_results]
}

summary_path = os.path.join(OUTPUT_DIR, 'final_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Results saved to: {OUTPUT_DIR}")
print(f"  • final_summary.json")
print(f"  • fold_*_test_metrics.json")
print(f"  • best_model_fold*.pth")